# WASB fine-tune — SMOKE TEST (diagnóstico primero)

Objetivo de ESTE notebook: confirmar que las piezas se enchufan — el modelo
HRNet(wasb) carga los pesos soccer, el dataloader entrega (imgs, heatmaps target)
sobre NUESTRO dataset, el modelo produce salida, la QFL computa y el loss baja en
unos batches. NO es el fine-tune final; es el andamio.

⚠️ El trainer del repo (`train_and_test.py`) está incompleto (`assert 0`), así que
reusamos sus builders (build_model, build_dataloader, build_loss_criteria) y
escribimos el loop nosotros. Las celdas 3-4 son DIAGNÓSTICAS: imprimen shapes y
tipos reales para no adivinar. Pegame su salida y con eso finalizo el loop (celda 5).

## 1) Setup (repo WASB + nuestro repo + pesos + parches)

In [ ]:
import os, subprocess
if not os.path.exists('/content/WASB-SBDT'):
    !git clone -q https://github.com/nttcom/WASB-SBDT.git /content/WASB-SBDT
if not os.path.exists('/content/ncf_event_tracker'):
    !git clone -q --branch events-model https://github.com/pipachiesa/ncf_event_tracker.git /content/ncf_event_tracker
!cd /content/ncf_event_tracker && git pull -q origin events-model
!pip install -q hydra-core omegaconf gdown
W='/content/pretrained_weights/wasb_soccer_best.pth.tar'
os.makedirs('/content/pretrained_weights', exist_ok=True)
import gdown
if not os.path.exists(W):
    gdown.download(id='1pg0MpMtKZ6ziYEr4oyfKYPOO3hjLw94l', output=W, quiet=True)
# parche numpy 2.0
subprocess.run(r"grep -rl 'np\.Inf' /content/WASB-SBDT/src | xargs -r sed -i 's/np\.Inf/np.inf/g'", shell=True)
subprocess.run(r"grep -rl 'np\.NaN' /content/WASB-SBDT/src | xargs -r sed -i 's/np\.NaN/np.nan/g'", shell=True)
from google.colab import drive; drive.mount('/content/drive')
print('setup OK, peso:', os.path.exists(W))

## 2) Construir el dataset WASB desde el ball-GT de brasil
Train = primeras ~70% de las ventanas, val = últimas ~30% (split temporal, sin solape).

In [ ]:
VIDEO='/content/drive/MyDrive/football_analytics/videos/brasil_noruega.mp4'
assert os.path.exists(VIDEO), f'FALTA el video en Drive: {VIDEO}'
ROOT='/content/wasb_ft/soccer'
!cd /content/ncf_event_tracker && python3 events_model/make_wasb_dataset.py \
    --video "{VIDEO}" \
    --labels events_model/dataset/ball_gt/brasil_noruega_ball_labels.csv \
    --out {ROOT} --clip brasil --stride 2
print('frames:', len(os.listdir(f'{ROOT}/frames/brasil')))

## 3) DIAGNÓSTICO A: descubrir builders + cargar modelo y pesos
Imprime qué imports existen, carga HRNet(wasb) y el checkpoint (missing/unexpected keys).

In [ ]:
import sys; sys.path.insert(0,'/content/WASB-SBDT/src')
import torch, importlib, pkgutil
# descubrir de dónde salen los builders
for modname in ['models','dataloaders','datasets','losses','utils','runners']:
    try:
        m=importlib.import_module(modname)
        fns=[a for a in dir(m) if a.startswith('build_')]
        print(f'{modname}: {fns}')
    except Exception as e:
        print(f'{modname}: import falló ({type(e).__name__}: {e})')
# componer cfg con hydra
from hydra import compose, initialize_config_dir
from omegaconf import OmegaConf
with initialize_config_dir(version_base=None, config_dir='/content/WASB-SBDT/src/configs'):
    cfg = compose(config_name='eval', overrides=[
        'dataset=soccer','model=wasb','loss=qfl' if os.path.exists('/content/WASB-SBDT/src/configs/loss/qfl.yaml') else 'model=wasb',
        f'dataset.root_dir={ROOT}','dataset.train.videos=[brasil]','dataset.test.videos=[brasil]',
        'runner.gpus=[0]','detector.model_path='+W])
print('\\nCONFIG keys:', list(cfg.keys()))
print('model.name:', cfg.model.name, '| loss:', cfg.get('loss'))
from models import build_model
model=build_model(cfg).cuda()
ck=torch.load(W, map_location='cpu')
print('\\ncheckpoint keys:', list(ck.keys())[:5])
sd=ck['model_state_dict']
res=model.load_state_dict(sd, strict=False)
print('missing keys:', len(res.missing_keys), '| unexpected:', len(res.unexpected_keys))
print('  (0/0 = carga perfecta)')

## 4) DIAGNÓSTICO B: un batch del dataloader + salida del modelo
Imprime shapes/tipos reales de (imgs, hms) y de model(imgs) — esto define el loss.

In [ ]:
from dataloaders import build_dataloader
loaders = build_dataloader(cfg)
print('build_dataloader devolvió', len(loaders), 'loaders')
train_loader = loaders[0]
batch = next(iter(train_loader))
print('tipos del batch:', [type(x).__name__ for x in batch])
for i,x in enumerate(batch):
    if torch.is_tensor(x): print(f'  [{i}] tensor shape {tuple(x.shape)} dtype {x.dtype}')
    else: print(f'  [{i}] {type(x).__name__} len {len(x) if hasattr(x,"__len__") else "?"}')
imgs=batch[0].cuda()
model.eval()
with torch.no_grad(): out=model(imgs)
print('\\nmodel(imgs) tipo:', type(out).__name__)
if torch.is_tensor(out): print('  salida tensor shape', tuple(out.shape))
elif isinstance(out,(list,tuple)): print('  salida lista de', len(out), '->', [tuple(o.shape) if torch.is_tensor(o) else type(o).__name__ for o in out])
print('\\n>>> PEGAME TODA ESTA SALIDA (celdas 3 y 4) y finalizo el loop de entrenamiento (QFL + optimizer + val).')

## 5) (pendiente) Loop de fine-tune
Lo completo con los shapes reales de las celdas 3-4: QFL(model(imgs), hms) + optimizer
+ split temporal + val con acc@100 contra el ball-GT. No lo corras hasta que 3-4 pasen.

In [ ]:
print('Esperando la salida de las celdas 3 y 4 para finalizar el loop.')